In [5]:
import sys
import os

# Add the parent directory (src) to the system path
# The '..' tells it to look one folder up from where the notebook is currently running
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

In [6]:
from ingest import load_faq_data, build_index
from dotenv import load_dotenv
from openai import OpenAI
load_dotenv()

True

In [7]:
data = load_faq_data()
index = build_index(data)

In [8]:
client = OpenAI(
  base_url="https://openrouter.ai/api/v1",
  api_key=os.getenv("OPENROUTER_API_KEY"),
)

In [9]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = client.responses.create(
    model="gpt-4o-mini",
    input=messages,
)

response.output_text

"Whether you can join a course depends on various factors such as the course's enrollment policies, deadlines, and prerequisites. If it's an online course, check the course provider's website for registration details. If it's a university course, you may need to contact the admissions office or the course instructor directly. If you provide more details about the course, I may be able to offer more specific guidance!"

In [10]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

### Create  a function tool schema

In [11]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [12]:
response = client.responses.create(
    model="gpt-4o-mini",
    input=messages,
    tools=[search_tool],
)

response.output

[ResponseFunctionToolCall(arguments='{"query":"join course"}', call_id='call_DYuLo20SW6hyBtZp7Tg2QMIY', name='search', type='function_call', id='fc_tmp_qzcckxdvuck', namespace=None, status='completed')]

In [13]:
import json

call = response.output[0]
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)

### Now we send the result back to the LLM

In [14]:
messages.extend(response.output)

messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

In [15]:
response = client.responses.create(
    model="gpt-4o-mini",
    input=messages,
    tools=[search_tool],
)

response.output_text

'Yes, you can still join the course! However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted. You can start learning and submitting homework without needing prior registration. '

### Calculating the ussage cost of the LLM

In [16]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(757, 46)

In [17]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_gpt54mini_price(757, 53)
print("Total cost: $", round(result["total_cost"], 8))

Total cost: $ 0.00014535


### Building the Agent Loop

In [18]:
# Helper function to handle function calls from the model
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

#### Make a single Call

In [19]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

question = "I just discovered the course. Can I join it?"

In [21]:
#Send the call to the model with the instructions and the user question
messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

response = client.responses.create(
    model="gpt-4o-mini",
    input=messages,
    tools=[search_tool],
)

messages.extend(response.output)
has_function_calls = False

#Check the response for function calls and handle them
for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

function_call: search {"query":"join course enrollment"}


#### Write the full agent loop

In [22]:
it = 1

while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = client.responses.create(
        model="gpt-4o-mini",
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    it = it + 1
    if has_function_calls == False:
        break

iteration #1...
ASSISTANT:
Yes, you can still join the course! If you want to receive a certificate, make sure to submit your project while the submissions are still being accepted.

Additionally, you don't need a confirmation email for your enrollment; you're accepted right away, and you can start learning and submitting homework without formal registration while the submission forms are open. 

Is there anything else you would like to know about the course?


#### Wrap the whole agentic loop

In [23]:
def agent_loop(instructions, question, model="gpt-4o-mini") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [24]:
agent_loop(instructions, "How do I run Olama locally?")

iteration #1...
function_call: search {"query":"run Olama locally"}
iteration #2...
ASSISTANT:
To run Olama locally, you'll need to follow these steps for installation and execution:

1. **Install Olama:**
   - Visit [https://ollama.com/download](https://ollama.com/download) and choose the appropriate installer for your operating system:
     - **macOS**: Download the `.pkg` file and install it.
     - **Windows**: Download the `.msi` file and install it.
     - **Linux**: Run the following command in the terminal:
       ```bash
       curl -fsSL https://ollama.com/install.sh | sh
       ```

2. **Run the LLaMA 3 Model:**
   - Once you have installed Olama, open a terminal and type the following command:
     ```bash
     ollama run llama3
     ```
   This will download the LLaMA 3 model (approximately 4GB) and start it locally, allowing you to interact with it via a chat-like interface.

3. **Test the Local Server:**
   - You can test if the Ollama local server is running by executin

'To run Olama locally, you\'ll need to follow these steps for installation and execution:\n\n1. **Install Olama:**\n   - Visit [https://ollama.com/download](https://ollama.com/download) and choose the appropriate installer for your operating system:\n     - **macOS**: Download the `.pkg` file and install it.\n     - **Windows**: Download the `.msi` file and install it.\n     - **Linux**: Run the following command in the terminal:\n       ```bash\n       curl -fsSL https://ollama.com/install.sh | sh\n       ```\n\n2. **Run the LLaMA 3 Model:**\n   - Once you have installed Olama, open a terminal and type the following command:\n     ```bash\n     ollama run llama3\n     ```\n   This will download the LLaMA 3 model (approximately 4GB) and start it locally, allowing you to interact with it via a chat-like interface.\n\n3. **Test the Local Server:**\n   - You can test if the Ollama local server is running by executing:\n     ```bash\n     curl http://localhost:11434\n     ```\n   If everyt

In [25]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "I just discovered the course. Can I join it?")

iteration #1...
function_call: search {"query":"join course enrollment information"}
iteration #2...
ASSISTANT:
Yes, you can still join the course! However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted.

You don't necessarily need to register; you can start learning and submitting homework without completing any registration process—it's mostly for gauging interest. 

If you have any other questions or need more information about specific parts of the course, feel free to ask!


"Yes, you can still join the course! However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted.\n\nYou don't necessarily need to register; you can start learning and submitting homework without completing any registration process—it's mostly for gauging interest. \n\nIf you have any other questions or need more information about specific parts of the course, feel free to ask!"

In [26]:
agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"Queen Gambit"}
iteration #2...
function_call: search {"query":"Queen Gambit chess"}
iteration #3...
function_call: search {"query": "Queen Gambit definition"}
function_call: search {"query": "Queen Gambit chess opening"}
iteration #4...
ASSISTANT:
The "Queen's Gambit" is a chess opening that begins with the moves 1. d4 d5 2. c4. It's named after the pawn move to c4, which offers a pawn to challenge the center and aims to control the game's tempo. 

Although I didn't find specific descriptions in the course FAQ, this opening is quite popular among players for providing dynamic play and chances for both sides.

It gained additional fame from the Netflix miniseries "The Queen's Gambit," which revolves around a young chess prodigy, highlighting the game's complexity and beauty.

Is there a specific aspect of the Queen's Gambit or chess in general that you'd like to explore further?


'The "Queen\'s Gambit" is a chess opening that begins with the moves 1. d4 d5 2. c4. It\'s named after the pawn move to c4, which offers a pawn to challenge the center and aims to control the game\'s tempo. \n\nAlthough I didn\'t find specific descriptions in the course FAQ, this opening is quite popular among players for providing dynamic play and chances for both sides.\n\nIt gained additional fame from the Netflix miniseries "The Queen\'s Gambit," which revolves around a young chess prodigy, highlighting the game\'s complexity and beauty.\n\nIs there a specific aspect of the Queen\'s Gambit or chess in general that you\'d like to explore further?'

In [27]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit"}
iteration #2...
ASSISTANT:
It seems that the query "queen gambit" is not related to the course or its logistics, and thus there's no available information in the FAQ database. If you have any other questions regarding the course, please feel free to ask! Are there other areas that you would like to explore?


'It seems that the query "queen gambit" is not related to the course or its logistics, and thus there\'s no available information in the FAQ database. If you have any other questions regarding the course, please feel free to ask! Are there other areas that you would like to explore?'